# Podcast ➜ 10-Minute Roundtable Recap (Diarized + TTS)

This notebook processes a long podcast audio file (>=1 hour recommended) and produces:

1. A speaker-labeled transcript with timestamps.
2. A concise, multi-speaker “roundtable recap” script (~10 minutes spoken).
3. A synthesized recap audio file at `output/recap_10min.mp3` using **synthetic voices**.

## Outputs
- `output/transcripts/full_transcript.json`
- `output/transcripts/full_transcript.txt`
- `output/recap_script.txt`
- `output/recap_script.json`
- `output/recap_10min.mp3`

## Notes on diarization support
- This notebook **prefers** a diarizing transcription model (for example `gpt-4o-transcribe-diarize`) and falls back to non-diarizing models if unavailable.
- Diarization fields and response shape can vary by model/version; defensive parsing is included.

## Cost, privacy, and limitations
- API costs can be significant for long audio + multiple TTS turns.
- Audio/text is sent to OpenAI APIs; review your data governance requirements.
- Speaker mapping across chunks is heuristic when stable cross-chunk IDs are not guaranteed.
- Very noisy/crosstalk-heavy recordings may reduce diarization quality.

## Safety / legal note
- Only process audio you have rights/consent to process.
- This notebook uses synthetic TTS voices and **does not clone** real speakers.
- Do not impersonate real people or create deceptive voice content.

In [ ]:
# If using Colab, uncomment the apt line if ffmpeg is missing.
# !apt-get update -qq && apt-get install -y ffmpeg

%pip -q install --upgrade openai pydub ffmpeg-python tqdm python-dotenv

In [ ]:
import os
import io
import re
import json
import math
import time
import shutil
import random
import tempfile
import subprocess
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

from dotenv import load_dotenv
from tqdm.auto import tqdm
from pydub import AudioSegment
from pydub.effects import normalize
from IPython.display import Audio, display

from openai import OpenAI

load_dotenv()

# -------------------- USER CONFIG --------------------
INPUT_AUDIO_PATH = "input/podcast.mp3"      # path to your long podcast file
OUTPUT_DIR = "output"
CHUNK_MINUTES = 15                           # recommended 10-20
TARGET_MINUTES = 10                          # target recap length
DRY_RUN = False                              # True => transcribe only first chunk

TRANSCRIBE_MODEL_PREFERRED = "gpt-4o-transcribe-diarize"
TRANSCRIBE_MODEL_FALLBACKS = [
    "gpt-4o-transcribe",
    "gpt-4o-mini-transcribe",
]

# Use a stable alias/model available in your account.
TTS_MODEL = "gpt-4o-mini-tts"
TEXT_MODEL = "gpt-4.1-mini"

MAX_TOKENS_SCRIPT = 2200
TWO_SPEAKER_MODE = True
# You may need to swap SPEAKER_1 and SPEAKER_2 labels depending on diarizer assignment in your audio.
CANONICAL_SPEAKER_NAMES = {"SPEAKER_1": "Joe Rogan", "SPEAKER_2": "Bernie Sanders", "OTHER": "Other/Clip"}
MAX_RETRIES = 5
BACKOFF_BASE_SEC = 1.5

# Candidate synthetic voices (API/account availability can vary).
VOICE_POOL = ["alloy", "verse", "coral", "ember", "sage", "breeze", "mist"]

# Timing heuristics
WORDS_PER_MIN = 145
MIN_WORDS_TARGET = int(9 * WORDS_PER_MIN)
MAX_WORDS_TARGET = int(11 * WORDS_PER_MIN)
IDEAL_WORDS_TARGET = int(TARGET_MINUTES * WORDS_PER_MIN)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
assert OPENAI_API_KEY, "Set OPENAI_API_KEY in your environment before running."

client = OpenAI(api_key=OPENAI_API_KEY)

OUT = Path(OUTPUT_DIR)
TRANSCRIPTS_DIR = OUT / "transcripts"
TRANSCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def ensure_ffmpeg() -> str:
    """Return ffmpeg path or raise helpful error."""
    ffmpeg_path = shutil.which("ffmpeg")
    if ffmpeg_path:
        return ffmpeg_path
    raise RuntimeError(
        "ffmpeg not found. Install ffmpeg (e.g., apt-get install ffmpeg) and rerun."
    )


def load_audio_metadata(input_path: str) -> Dict[str, Any]:
    p = Path(input_path)
    if not p.exists():
        raise FileNotFoundError(f"Audio file not found: {p}")
    audio = AudioSegment.from_file(p)
    duration_sec = len(audio) / 1000.0
    return {
        "path": str(p),
        "duration_sec": duration_sec,
        "duration_min": duration_sec / 60.0,
        "channels": audio.channels,
        "frame_rate": audio.frame_rate,
        "sample_width": audio.sample_width,
    }


def split_audio_to_chunks(input_path: str, chunk_minutes: int) -> List[Dict[str, Any]]:
    """
    Splits audio into fixed-size chunks and writes chunk files to output/chunks.
    Returns list of dicts:
      {"chunk_index": i, "path": ..., "start_sec": ..., "end_sec": ...}
    """
    chunk_dir = OUT / "chunks"
    chunk_dir.mkdir(parents=True, exist_ok=True)

    audio = AudioSegment.from_file(input_path)
    total_ms = len(audio)
    chunk_ms = int(chunk_minutes * 60 * 1000)

    chunks = []
    idx = 0
    for start_ms in range(0, total_ms, chunk_ms):
        end_ms = min(start_ms + chunk_ms, total_ms)
        chunk_audio = audio[start_ms:end_ms]
        chunk_path = chunk_dir / f"chunk_{idx:03d}.mp3"
        chunk_audio.export(chunk_path, format="mp3", bitrate="128k")
        chunks.append(
            {
                "chunk_index": idx,
                "path": str(chunk_path),
                "start_sec": start_ms / 1000.0,
                "end_sec": end_ms / 1000.0,
            }
        )
        idx += 1
    return chunks


def _extract_segments_from_transcription(resp: Any) -> List[Dict[str, Any]]:
    """Best-effort parser for varying transcription response shapes."""
    if resp is None:
        return []

    # Convert Pydantic-ish object to dict where possible
    if hasattr(resp, "model_dump"):
        data = resp.model_dump()
    elif isinstance(resp, dict):
        data = resp
    else:
        try:
            data = json.loads(str(resp))
        except Exception:
            data = {}

    segments = []

    # Common pattern: data['segments']
    raw_segments = data.get("segments") if isinstance(data, dict) else None
    if isinstance(raw_segments, list):
        for s in raw_segments:
            if not isinstance(s, dict):
                continue
            segments.append(
                {
                    "start": float(s.get("start", 0.0)),
                    "end": float(s.get("end", s.get("start", 0.0))),
                    "speaker": str(s.get("speaker", s.get("speaker_label", "UNKNOWN"))),
                    "text": str(s.get("text", "")).strip(),
                }
            )

    # Alternate pattern: data['words'] with speaker tags -> group words into pseudo-segments
    if not segments and isinstance(data, dict) and isinstance(data.get("words"), list):
        words = data["words"]
        current = None
        for w in words:
            if not isinstance(w, dict):
                continue
            speaker = str(w.get("speaker", w.get("speaker_label", "UNKNOWN")))
            token = str(w.get("word", w.get("text", ""))).strip()
            start = float(w.get("start", 0.0))
            end = float(w.get("end", start))
            if not token:
                continue
            if current is None or current["speaker"] != speaker:
                if current:
                    segments.append(current)
                current = {"start": start, "end": end, "speaker": speaker, "text": token}
            else:
                current["end"] = end
                current["text"] += " " + token
        if current:
            segments.append(current)

    # Fallback: whole text as one segment
    if not segments:
        text = ""
        if isinstance(data, dict):
            text = str(data.get("text", "")).strip()
        if text:
            segments = [{"start": 0.0, "end": 0.0, "speaker": "UNKNOWN", "text": text}]

    return [s for s in segments if s.get("text")]


def _extract_error_text(e: Exception) -> str:
    pieces = [str(e)]
    body = getattr(e, "body", None)
    if body:
        pieces.append(str(body))
    resp = getattr(e, "response", None)
    if resp is not None:
        try:
            pieces.append(str(resp.text))
        except Exception:
            pass
    return " ".join([p for p in pieces if p])


def _should_retry_exception(e: Exception) -> bool:
    """Retry transient failures, but not obvious request-shape errors."""
    text = _extract_error_text(e).lower()
    non_retry_markers = [
        "invalid_request_error",
        "unsupported_value",
        "response_format",
        "not compatible with model",
        "timestamp_granularities",
    ]
    return not any(m in text for m in non_retry_markers)


def call_with_retries(fn, *args, **kwargs):
    for attempt in range(MAX_RETRIES):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            if attempt == MAX_RETRIES - 1 or not _should_retry_exception(e):
                raise
            sleep_s = BACKOFF_BASE_SEC * (2 ** attempt) + random.random()
            print(f"Retry {attempt + 1}/{MAX_RETRIES} after error: {e} | sleeping {sleep_s:.1f}s")
            time.sleep(sleep_s)


def transcribe_chunk(path: str, model: str) -> Dict[str, Any]:
    """
    Returns structured transcript dict:
    {
      "model":..., "response_format":..., "segments":[{"start":..,"end":..,"speaker":..,"text":..}], "raw":...
    }

    Notes:
    - Diarization models require response_format='diarized_json' and chunking_strategy='auto'.
    - We then attempt additional compatible request shapes in descending richness.
    """
    is_diarize_model = "diarize" in model.lower()
    if is_diarize_model:
        # Diarization models: do NOT try verbose_json; keep chunking_strategy on each attempt.
        attempts = [
            {"response_format": "diarized_json", "extra_body": {"chunking_strategy": "auto"}},
            {"response_format": "json", "extra_body": {"chunking_strategy": "auto"}},
            {"response_format": "text", "extra_body": {"chunking_strategy": "auto"}},
        ]
    else:
        attempts = [
            {"response_format": "verbose_json", "timestamp_granularities": ["segment"]},
            {"response_format": "json"},
            {"response_format": "text"},
        ]
    last_err = None

    for req in attempts:
        try:
            with open(path, "rb") as f:
                resp = client.audio.transcriptions.create(
                    model=model,
                    file=f,
                    **req,
                )
            segments = _extract_segments_from_transcription(resp)
            raw = resp.model_dump() if hasattr(resp, "model_dump") else (resp if isinstance(resp, dict) else {"repr": str(resp)})
            return {
                "model": model,
                "response_format": req["response_format"],
                "segments": segments,
                "raw": raw,
            }
        except Exception as e:
            last_err = e
            msg = _extract_error_text(e).lower()
            is_shape_error = any(k in msg for k in ["unsupported_value", "not compatible with model", "response_format", "timestamp_granularities", "chunking_strategy"])
            if is_shape_error:
                print(f"Model {model}: request shape unsupported for {req['response_format']}, trying next format.")
                continue
            raise

    raise RuntimeError(f"All response_format attempts failed for model {model}. Last error: {last_err}")


def try_transcribe_with_fallback(path: str, preferred: str, fallbacks: List[str]) -> Dict[str, Any]:
    models = [preferred] + list(fallbacks)
    last_err = None
    for m in models:
        try:
            result = call_with_retries(transcribe_chunk, path, m)
            if m != preferred:
                print(f"Used fallback transcription model: {m}")
            return result
        except Exception as e:
            print(f"Model failed: {m} -> {e}")
            last_err = e
    raise RuntimeError(f"All transcription models failed. Last error: {last_err}")


def merge_transcripts(chunk_transcripts: List[Dict[str, Any]]) -> Dict[str, Any]:
    merged_segments = []
    for ch in chunk_transcripts:
        offset = float(ch["start_sec"])
        for s in ch["segments"]:
            merged_segments.append(
                {
                    "start": float(s.get("start", 0.0)) + offset,
                    "end": float(s.get("end", 0.0)) + offset,
                    "speaker": str(s.get("speaker", "UNKNOWN")),
                    "text": str(s.get("text", "")).strip(),
                    "chunk_index": ch["chunk_index"],
                }
            )
    merged_segments = [s for s in merged_segments if s["text"]]
    merged_segments.sort(key=lambda x: (x["start"], x["end"]))
    return {"segments": merged_segments}


def _norm_text(t: str) -> set:
    toks = re.findall(r"[a-z0-9']+", t.lower())
    return set(toks)


def _jaccard(a: str, b: str) -> float:
    sa, sb = _norm_text(a), _norm_text(b)
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / max(1, len(sa | sb))


def normalize_speaker_labels(global_transcript: Dict[str, Any]) -> Dict[str, Any]:
    """
    Attempts to normalize speaker labels across chunks.

    Strategy:
      1) If labels already look global (few repeating labels), keep with canonical SPEAKER_n mapping.
      2) If chunk-local labels vary, map labels in chunk k to previous global speakers using
         similarity between boundary segment text + local co-occurrence frequencies.
      3) Fallback: assign new SPEAKER_n when uncertain and mark instability warning.
    """
    segments = global_transcript.get("segments", [])
    if not segments:
        return {"segments": [], "speaker_map": {}, "warnings": ["No segments found."]}

    by_chunk = {}
    for s in segments:
        by_chunk.setdefault(s["chunk_index"], []).append(s)

    for k in by_chunk:
        by_chunk[k] = sorted(by_chunk[k], key=lambda x: (x["start"], x["end"]))

    speaker_map = {}  # (chunk_idx, local_label) -> SPEAKER_n
    next_id = 1
    warnings = []

    def get_or_new(chunk_idx: int, local_label: str) -> str:
        nonlocal next_id
        key = (chunk_idx, local_label)
        if key in speaker_map:
            return speaker_map[key]
        sid = f"SPEAKER_{next_id}"
        next_id += 1
        speaker_map[key] = sid
        return sid

    chunk_ids = sorted(by_chunk.keys())
    first = chunk_ids[0]
    first_labels = sorted({s["speaker"] for s in by_chunk[first]})
    for lbl in first_labels:
        get_or_new(first, lbl)

    for i in range(1, len(chunk_ids)):
        prev_c = chunk_ids[i - 1]
        cur_c = chunk_ids[i]

        prev_tail = by_chunk[prev_c][-8:] if len(by_chunk[prev_c]) > 8 else by_chunk[prev_c]
        cur_head = by_chunk[cur_c][:8] if len(by_chunk[cur_c]) > 8 else by_chunk[cur_c]

        prev_by_global = {}
        for s in prev_tail:
            g = speaker_map.get((prev_c, s["speaker"]))
            if not g:
                g = get_or_new(prev_c, s["speaker"])
            prev_by_global.setdefault(g, []).append(s["text"])

        cur_by_local = {}
        for s in cur_head:
            cur_by_local.setdefault(s["speaker"], []).append(s["text"])

        assigned_globals = set()
        for local_lbl, texts in cur_by_local.items():
            cur_text = " ".join(texts)
            best_g = None
            best_score = 0.0
            for g, ptexts in prev_by_global.items():
                if g in assigned_globals:
                    continue
                score = _jaccard(cur_text, " ".join(ptexts))
                if score > best_score:
                    best_score = score
                    best_g = g

            # Conservative threshold: if weak similarity, create new speaker
            if best_g and best_score >= 0.08:
                speaker_map[(cur_c, local_lbl)] = best_g
                assigned_globals.add(best_g)
            else:
                get_or_new(cur_c, local_lbl)

    normalized = []
    for s in segments:
        g = speaker_map.get((s["chunk_index"], s["speaker"]))
        if not g:
            warnings.append("Unstable mapping encountered; applied fallback remap.")
            g = get_or_new(s["chunk_index"], s["speaker"])
        normalized.append({**s, "speaker": g})

    normalized.sort(key=lambda x: (x["start"], x["end"]))

    if any("UNKNOWN" in str(s["speaker"]) for s in segments):
        warnings.append("Some segments lacked diarization labels; mapping may be less stable.")

    return {
        "segments": normalized,
        "speaker_map": {f"chunk{c}:{l}": g for (c, l), g in speaker_map.items()},
        "warnings": sorted(set(warnings)),
    }


def canonicalize_two_speakers(full_transcript: Dict[str, Any]) -> Dict[str, Any]:
    """Collapse normalized labels into SPEAKER_1, SPEAKER_2, OTHER using total talk time."""
    segments = full_transcript.get("segments", [])
    if not segments:
        return {
            **full_transcript,
            "canonicalization": {
                "mode": "two_speaker",
                "top2_original_labels": [],
                "original_label_to_canonical": {},
                "talk_time_sec_by_original": {},
            },
        }

    talk_time_sec_by_original = {}
    for seg in segments:
        original = str(seg.get("speaker", "UNKNOWN"))
        dur = max(0.0, float(seg.get("end", 0.0)) - float(seg.get("start", 0.0)))
        talk_time_sec_by_original[original] = talk_time_sec_by_original.get(original, 0.0) + dur

    ranked = sorted(talk_time_sec_by_original.items(), key=lambda kv: kv[1], reverse=True)
    top2_original_labels = [label for label, _ in ranked[:2]]

    original_label_to_canonical = {}
    for i, label in enumerate(top2_original_labels, start=1):
        original_label_to_canonical[label] = f"SPEAKER_{i}"
    for label in talk_time_sec_by_original.keys():
        if label not in original_label_to_canonical:
            original_label_to_canonical[label] = "OTHER"

    canonical_segments = []
    for seg in segments:
        original = str(seg.get("speaker", "UNKNOWN"))
        canonical_segments.append({
            **seg,
            "original_speaker": original,
            "speaker": original_label_to_canonical.get(original, "OTHER"),
        })

    warnings = list(full_transcript.get("warnings", []))
    warnings.append("Two-speaker canonicalization enabled: collapsed labels into SPEAKER_1, SPEAKER_2, OTHER.")

    return {
        **full_transcript,
        "segments": canonical_segments,
        "warnings": sorted(set(warnings)),
        "canonicalization": {
            "mode": "two_speaker",
            "top2_original_labels": top2_original_labels,
            "original_label_to_canonical": original_label_to_canonical,
            "talk_time_sec_by_original": talk_time_sec_by_original,
        },
    }


def _fmt_ts(sec: float) -> str:
    sec = max(0, int(sec))
    h = sec // 3600
    m = (sec % 3600) // 60
    s = sec % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


def write_readable_transcript_txt(full_transcript: Dict[str, Any], out_path: Path) -> None:
    lines = []
    for seg in full_transcript.get("segments", []):
        lines.append(f"[{_fmt_ts(seg['start'])} - {_fmt_ts(seg['end'])}] {seg['speaker']}: {seg['text']}")
    out_path.write_text("\n".join(lines), encoding="utf-8")

In [ ]:
# ----- Transcription pipeline -----
ensure_ffmpeg()
meta = load_audio_metadata(INPUT_AUDIO_PATH)
print("Audio metadata:", json.dumps(meta, indent=2))

chunks = split_audio_to_chunks(INPUT_AUDIO_PATH, CHUNK_MINUTES)
if DRY_RUN:
    chunks = chunks[:1]
    print("DRY_RUN enabled: only first chunk will be transcribed.")

chunk_results = []
for ch in tqdm(chunks, desc="Transcribing chunks"):
    tr = try_transcribe_with_fallback(
        ch["path"],
        preferred=TRANSCRIBE_MODEL_PREFERRED,
        fallbacks=TRANSCRIBE_MODEL_FALLBACKS,
    )
    chunk_payload = {
        "chunk_index": ch["chunk_index"],
        "start_sec": ch["start_sec"],
        "end_sec": ch["end_sec"],
        "model": tr["model"],
        "segments": tr["segments"],
        "raw": tr["raw"],
    }
    chunk_results.append(chunk_payload)

    with open(TRANSCRIPTS_DIR / f"chunk_{ch['chunk_index']:03d}.json", "w", encoding="utf-8") as f:
        json.dump(chunk_payload, f, ensure_ascii=False, indent=2)

merged = merge_transcripts(chunk_results)
normalized = normalize_speaker_labels(merged)

full_transcript = {
    "metadata": {
        "input_audio": str(INPUT_AUDIO_PATH),
        "duration_sec": meta["duration_sec"],
        "chunk_minutes": CHUNK_MINUTES,
        "dry_run": DRY_RUN,
        "two_speaker_mode": TWO_SPEAKER_MODE,
    },
    "warnings": normalized.get("warnings", []),
    "speaker_map": normalized.get("speaker_map", {}),
    "segments": normalized["segments"],
}
if TWO_SPEAKER_MODE:
    full_transcript = canonicalize_two_speakers(full_transcript)

full_json_path = TRANSCRIPTS_DIR / "full_transcript.json"
with open(full_json_path, "w", encoding="utf-8") as f:
    json.dump(full_transcript, f, ensure_ascii=False, indent=2)

full_txt_path = TRANSCRIPTS_DIR / "full_transcript.txt"
write_readable_transcript_txt(full_transcript, full_txt_path)

print(f"Saved: {full_json_path}")
print(f"Saved: {full_txt_path}")
if full_transcript["warnings"]:
    print("Warnings:")
    for w in full_transcript["warnings"]:
        print(" -", w)

In [ ]:

def _collect_speakers(full_transcript: Dict[str, Any]) -> List[str]:
    if TWO_SPEAKER_MODE:
        return ["SPEAKER_1", "SPEAKER_2", "OTHER"]
    sp = sorted({s["speaker"] for s in full_transcript.get("segments", [])})
    return [x for x in sp if x.startswith("SPEAKER_")] or sp


def _transcript_for_prompt(full_transcript: Dict[str, Any], max_chars: int = 120000) -> str:
    lines = []
    for s in full_transcript.get("segments", []):
        lines.append(f"[{_fmt_ts(s['start'])}] {s['speaker']}: {s['text']}")
    text = "\n".join(lines)
    return text[:max_chars]


def _fallback_script_from_excerpt(transcript_excerpt: str, speakers: List[str]) -> Dict[str, Any]:
    raw_lines = [ln.strip() for ln in transcript_excerpt.splitlines() if ":" in ln]
    if not raw_lines:
        raw_lines = ["[00:00:00] OTHER: Transcript unavailable for fallback summary."]

    target_n = min(24, max(12, len(raw_lines) // 20 if len(raw_lines) > 20 else 12))
    picks = []
    if len(raw_lines) <= target_n:
        picks = raw_lines
    else:
        for i in range(target_n):
            idx = int(i * (len(raw_lines) - 1) / max(1, target_n - 1))
            picks.append(raw_lines[idx])

    lines = []
    alt = ["SPEAKER_1", "SPEAKER_2"] if "SPEAKER_1" in speakers and "SPEAKER_2" in speakers else speakers[:2]
    if not alt:
        alt = ["OTHER"]

    for i, ln in enumerate(picks):
        payload = ln.split(":", 1)[-1].strip()
        payload = re.sub(r"\s+", " ", payload)
        if not payload:
            payload = "Key point noted in the conversation."
        text = f"Key point: {payload[:240]}"
        spk = alt[i % len(alt)] if alt else "OTHER"
        lines.append({"speaker": spk, "text": text})

    while len(lines) < 10:
        i = len(lines)
        spk = alt[i % len(alt)] if alt else "OTHER"
        lines.append({"speaker": spk, "text": "Additional recap point from the discussion context."})

    wc = sum(len(x["text"].split()) for x in lines)
    return {
        "script_lines": lines,
        "estimated_word_count": wc,
        "coverage_notes": "Python fallback script generated from transcript excerpt.",
    }


def generate_roundtable_script(full_transcript: Dict[str, Any]) -> Dict[str, Any]:
    speakers = _collect_speakers(full_transcript)
    transcript_excerpt = _transcript_for_prompt(full_transcript)

    system = (
        "You create faithful, concise multi-speaker recap dialogues from transcripts. "
        "Never add facts not present in source. Use hedges when uncertain."
    )

    user = f"""
Create a high-density recap dialogue from the transcript below.

Requirements:
- Allowed speaker labels: {', '.join(speakers)}
- Keep factual fidelity to source; do NOT invent details.
- Include key points, trade-offs, and brief clarifications/disagreements if present.
- Target total spoken length around {TARGET_MINUTES} minutes.
- Aim for total words in [{MIN_WORDS_TARGET}, {MAX_WORDS_TARGET}], ideal ~{IDEAL_WORDS_TARGET}.
- Speaker for each line must be one of the allowed labels; if unsure use OTHER.

Return STRICT JSON only (no markdown/code fences):
{{
  "script_lines": [{{"speaker": "SPEAKER_1|SPEAKER_2|OTHER", "text": "..."}}],
  "estimated_word_count": 1234,
  "coverage_notes": "..."
}}

Transcript:
{transcript_excerpt}
"""

    resp = call_with_retries(
        client.responses.create,
        model=TEXT_MODEL,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        max_output_tokens=MAX_TOKENS_SCRIPT,
    )

    out_text = ""
    if hasattr(resp, "output_text") and resp.output_text:
        out_text = resp.output_text
    else:
        try:
            out_text = json.dumps(resp.model_dump(), ensure_ascii=False)
        except Exception:
            out_text = str(resp)

    script_data = None
    try:
        script_data = json.loads(out_text)
    except Exception:
        m = re.search(r"\{[\s\S]*\}", out_text)
        if m:
            try:
                script_data = json.loads(m.group(0))
            except Exception:
                script_data = None

    if not isinstance(script_data, dict):
        script_data = {"script_lines": [], "estimated_word_count": 0, "coverage_notes": "Model JSON parse failed."}

    valid = set(speakers)
    cleaned = []
    for item in script_data.get("script_lines", []):
        spk = str(item.get("speaker", "OTHER")).strip()
        txt = str(item.get("text", "")).strip()
        if not txt:
            continue
        if spk not in valid:
            spk = "OTHER" if "OTHER" in valid else (speakers[0] if speakers else "SPEAKER_1")
        cleaned.append({"speaker": spk, "text": txt})

    if not cleaned:
        script_data = _fallback_script_from_excerpt(transcript_excerpt, speakers)
        cleaned = script_data["script_lines"]

    wc = sum(len(x["text"].split()) for x in cleaned)
    if wc <= 0:
        script_data = _fallback_script_from_excerpt(transcript_excerpt, speakers)
        cleaned = script_data["script_lines"]
        wc = sum(len(x["text"].split()) for x in cleaned)

    script_data["script_lines"] = cleaned
    script_data["estimated_word_count"] = wc
    script_data["speakers"] = speakers
    script_data.setdefault("coverage_notes", "")
    return script_data


def build_recap_timeline(script_lines: List[Dict[str, str]]) -> List[Dict[str, Any]]:
    timeline = []
    t = 0.0
    wps = max(1e-6, WORDS_PER_MIN / 60.0)
    for item in script_lines:
        spk = str(item.get("speaker", "OTHER"))
        txt = str(item.get("text", "")).strip()
        words = len(txt.split())
        dur = max(2.0, words / wps)
        timeline.append(
            {
                "speaker_id": spk,
                "speaker_name": CANONICAL_SPEAKER_NAMES.get(spk, spk),
                "start_sec": round(t, 3),
                "end_sec": round(t + dur, 3),
                "text": txt,
            }
        )
        t += dur
    return timeline


script_data = generate_roundtable_script(full_transcript)

recap_txt = OUT / "recap_script.txt"
recap_json = OUT / "recap_script.json"
recap_timeline_path = OUT / "recap_timeline.json"

with open(recap_txt, "w", encoding="utf-8") as f:
    for x in script_data["script_lines"]:
        f.write(f"{x['speaker']}: {x['text']}\n")

with open(recap_json, "w", encoding="utf-8") as f:
    json.dump(script_data, f, ensure_ascii=False, indent=2)

recap_timeline = build_recap_timeline(script_data.get("script_lines", []))
with open(recap_timeline_path, "w", encoding="utf-8") as f:
    json.dump(recap_timeline, f, ensure_ascii=False, indent=2)

print(f"Saved: {recap_txt}")
print(f"Saved: {recap_json}")
print(f"Saved: {recap_timeline_path}")
print("Estimated words:", script_data.get("estimated_word_count"))


In [ ]:
def assign_voices_to_speakers(speakers: List[str]) -> Dict[str, str]:
    voice_map = {}
    for i, spk in enumerate(sorted(speakers)):
        voice_map[spk] = VOICE_POOL[i % len(VOICE_POOL)]
    return voice_map


def tts_line(text: str, voice: str, out_path: Path):
    """
    Synthesize one line to MP3.
    NOTE: TTS response shape may vary; this handles common binary-content patterns.
    """
    resp = call_with_retries(
        client.audio.speech.create,
        model=TTS_MODEL,
        voice=voice,
        input=text,
        format="mp3",
    )

    audio_bytes = None
    if hasattr(resp, "read"):
        audio_bytes = resp.read()
    elif hasattr(resp, "content"):
        audio_bytes = resp.content
    elif hasattr(resp, "audio") and isinstance(resp.audio, (bytes, bytearray)):
        audio_bytes = bytes(resp.audio)

    if not audio_bytes:
        raise RuntimeError("Unable to extract audio bytes from TTS response.")

    out_path.write_bytes(audio_bytes)


def estimate_duration_minutes(audio_path: Path) -> float:
    a = AudioSegment.from_file(audio_path)
    return len(a) / 1000.0 / 60.0


def rank_lines_for_importance(script_lines: List[Dict[str, str]], full_transcript: Dict[str, Any]) -> List[int]:
    """
    Returns line indices sorted from most important to least important.
    Uses model ranking; falls back to original order.
    """
    payload = [f"{i}: {x['speaker']}: {x['text']}" for i, x in enumerate(script_lines)]
    prompt = f"""
Rank these recap lines by importance to factual coverage (most important first).
Return strict JSON: {{"ranked_indices": [..]}}

Lines:
""" + "\n".join(payload)

    try:
        resp = call_with_retries(
            client.responses.create,
            model=TEXT_MODEL,
            input=prompt,
            max_output_tokens=600,
        )
        txt = resp.output_text if hasattr(resp, "output_text") else str(resp)
        data = json.loads(re.search(r"\{[\s\S]*\}", txt).group(0))
        ranked = data.get("ranked_indices", [])
        ranked = [i for i in ranked if isinstance(i, int) and 0 <= i < len(script_lines)]
        if len(ranked) == len(script_lines):
            return ranked
    except Exception:
        pass

    return list(range(len(script_lines)))


def speedup_audio_ffmpeg(in_path: Path, out_path: Path, factor: float = 1.05):
    # atempo supported roughly 0.5-2.0 per filter
    cmd = [
        ensure_ffmpeg(), "-y",
        "-i", str(in_path),
        "-filter:a", f"atempo={factor}",
        str(out_path),
    ]
    subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)


# Build voice map
speakers = script_data.get("speakers", [])
voice_map = assign_voices_to_speakers(speakers)
print("Voice map:", voice_map)
print("Safety: synthetic voices only; no real-voice cloning.")

# Render each line
lines = script_data.get("script_lines", [])
turn_dir = OUT / "tts_turns"
turn_dir.mkdir(parents=True, exist_ok=True)

pause = AudioSegment.silent(duration=350)
combined = AudioSegment.silent(duration=200)

for i, item in enumerate(tqdm(lines, desc="TTS rendering")):
    spk = item["speaker"]
    txt = item["text"].strip()
    if not txt:
        continue

    turn_mp3 = turn_dir / f"turn_{i:04d}.mp3"
    tts_line(txt, voice_map.get(spk, VOICE_POOL[0]), turn_mp3)
    seg = AudioSegment.from_file(turn_mp3)
    combined += seg + pause

# Optional loudness normalization
combined = normalize(combined)

raw_recap = OUT / "recap_10min_raw.mp3"
combined.export(raw_recap, format="mp3", bitrate="160k")

duration_min = estimate_duration_minutes(raw_recap)
print(f"Raw recap duration: {duration_min:.2f} min")

final_recap = OUT / "recap_10min.mp3"

if duration_min > 11:
    print("Duration > 11 min. Attempting content trim by importance...")
    ranked = rank_lines_for_importance(lines, full_transcript)

    # Keep top-N lines proportionally
    keep_ratio = max(0.6, 11.0 / duration_min)
    keep_n = max(1, int(len(lines) * keep_ratio))
    keep_set = set(ranked[:keep_n])

    trimmed = AudioSegment.silent(duration=200)
    for i, item in enumerate(lines):
        if i not in keep_set:
            continue
        turn_mp3 = turn_dir / f"turn_{i:04d}.mp3"
        if turn_mp3.exists():
            trimmed += AudioSegment.from_file(turn_mp3) + pause

    trimmed = normalize(trimmed)
    trimmed_path = OUT / "recap_10min_trimmed.mp3"
    trimmed.export(trimmed_path, format="mp3", bitrate="160k")
    duration_trimmed = estimate_duration_minutes(trimmed_path)
    print(f"Trimmed recap duration: {duration_trimmed:.2f} min")

    if duration_trimmed > 11:
        print("Still long; applying gentle 1.05x speed-up.")
        speedup_audio_ffmpeg(trimmed_path, final_recap, factor=1.05)
    else:
        shutil.copyfile(trimmed_path, final_recap)
else:
    shutil.copyfile(raw_recap, final_recap)

final_duration = estimate_duration_minutes(final_recap)
print(f"Final recap duration: {final_duration:.2f} min")

In [ ]:

final_recap = OUT / "recap_10min.mp3"
full_json = TRANSCRIPTS_DIR / "full_transcript.json"
full_txt = TRANSCRIPTS_DIR / "full_transcript.txt"
recap_txt = OUT / "recap_script.txt"
recap_json = OUT / "recap_script.json"
recap_timeline_path = OUT / "recap_timeline.json"

speaker_count = len({s["speaker"] for s in full_transcript.get("segments", [])})
canonical = full_transcript.get("canonicalization", {})
top2 = canonical.get("top2_original_labels", [])
timeline_est_sec = 0.0
if 'recap_timeline' in globals() and recap_timeline:
    timeline_est_sec = float(recap_timeline[-1].get("end_sec", 0.0))

summary_points = [
    f"Transcript segments: {len(full_transcript.get('segments', []))}",
    f"Speakers detected (post-canonicalization): {speaker_count}",
    f"Recap script words: {script_data.get('estimated_word_count', 0)}",
]

print("=== Final Report ===")
print(f"- Transcript JSON: {full_json}")
print(f"- Transcript TXT : {full_txt}")
print(f"- Recap script TXT: {recap_txt}")
print(f"- Recap script JSON: {recap_json}")
print(f"- Recap timeline JSON: {recap_timeline_path}")
print(f"- Recap audio MP3: {final_recap}")
print(f"- Recap duration: {estimate_duration_minutes(final_recap):.2f} min")
print(f"- Canonical mode enabled: {TWO_SPEAKER_MODE}")
print(f"- Top2 original labels chosen: {top2}")
print(f"- Recap timeline estimated duration (sec): {timeline_est_sec:.2f}")
print("- Bullet summary:")
for p in summary_points:
    print(f"  • {p}")

if final_recap.exists():
    display(Audio(str(final_recap)))
